In [ ]:
import torch
import torch.nn as nn
import timm
import numpy as np
import io
import time
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from peft import LoraConfig, get_peft_model
from transformers import BertTokenizer, BertModel
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
drive.mount('/content/drive')
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 336

CHECKPOINT_PATH = "/content/drive/MyDrive/Final-DinoV2/DinoV2_lora_mvtec.pt"
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
text_encoder = BertModel.from_pretrained("bert-base-uncased").to(DEVICE)
text_encoder.eval()

lora_cfg_dict = ckpt['lora_cfg']
base_model = timm.create_model(ckpt['model_name'], pretrained=True, num_classes=0, dynamic_img_size=True)

lora_config = LoraConfig(
    r=lora_cfg_dict['r'],
    lora_alpha=lora_cfg_dict['lora_alpha'],
    target_modules=lora_cfg_dict['target_modules'],
    lora_dropout=lora_cfg_dict['lora_dropout']
)

vit = get_peft_model(base_model, lora_config).to(DEVICE)
img_proj = nn.Linear(vit.model.num_features, text_encoder.config.hidden_size).to(DEVICE)

vit.load_state_dict(ckpt['vit_state'], strict=False)
img_proj.load_state_dict(ckpt['img_proj_state'])
logit_scale = nn.Parameter(ckpt.get('logit_scale', torch.tensor(np.log(1 / 0.07))).to(DEVICE))

vit.eval()
img_proj.eval()
print(f"Model restored successfully for: {ckpt['objects']}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model restored successfully for: ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


In [ ]:
file_upload = widgets.FileUpload(accept='.png, .jpg, .jpeg', multiple=False, description="Browse Image")
text_input = widgets.Text(value='gear, paper, wood', description='Classes:', style={'description_width': 'initial'})
run_button = widgets.Button(description='Classify', button_style='success', icon='search')
output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        clear_output()
        if not file_upload.value:
            print("Please upload an image first!")
            return

        candidate_labels = [label.strip() for label in text_input.value.split(',')]

        uploaded_file = list(file_upload.value.values())[0] if isinstance(file_upload.value, dict) else file_upload.value[0]
        img = Image.open(io.BytesIO(uploaded_file['content'])).convert("RGB")

        try:
            tfms = transforms.Compose([
                transforms.Resize((IMG_SIZE, IMG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
            img_tensor = tfms(img).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                img_feat = vit(img_tensor)
                img_feat = img_proj(img_feat)
                img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

                prompts = [f"a photo of a {label}" for label in candidate_labels]
                tokens = tokenizer(prompts, padding=True, return_tensors="pt").to(DEVICE)
                txt_feat = text_encoder(**tokens).pooler_output
                txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)

                logits = (img_feat @ txt_feat.t()) * (logit_scale.exp() * 1.5)
                probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
            ax1.imshow(img); ax1.set_title("Input Image"); ax1.axis('off')

            y_pos = range(len(candidate_labels))
            colors = ['#4CAF50' if i == probs.argmax() else '#757575' for i in range(len(candidate_labels))]
            bars = ax2.barh(y_pos, probs * 100, color=colors)
            ax2.set_yticks(y_pos)
            ax2.set_yticklabels([label.upper() for label in candidate_labels])
            ax2.invert_yaxis()
            ax2.set_xlabel('Probability (%)')
            ax2.set_xlim(0, 105)

            for bar in bars:
                ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{bar.get_width():.1f}%', va='center')

            plt.tight_layout()
            plt.show()
            print(f"WINNER: {candidate_labels[probs.argmax()].upper()}")

        except Exception as e:
            print(f"An error occurred: {e}")

run_button.on_click(on_button_clicked)
display(widgets.VBox([file_upload, text_input, run_button, output_area]))